# kinyarwanda tweet sentiment classification

fine tuning afroxlmr on kinyarwanda tweets and testing it against tfidf baselines

## 0 setup

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/mbonikevin/senti-kinyarwanda.git"
REPO_DIR = "senti-kinyarwanda"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40,<5", "accelerate>=0.30", "datasets>=2.18",
                    "sentencepiece", "protobuf"], check=True)

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

from src.data import load_splits, describe, save_processed, ID_TO_LABEL, LABEL_ORDER
from src.metrics import compute_metrics, format_metrics_report, save_metrics, plot_confusion_matrix, comparison_table
from src import viz, leakage
print("label mapping:", ID_TO_LABEL)

## 1 data

the kinyarwanda subset of afrisenti muhammad et al 2023 read from the official repo

In [ ]:
splits = load_splits() # light cleaning the urls > <url> whitespace
save_processed(splits)
describe(splits)

### 1.1 does the official split leak?

In [ ]:
print(leakage.overlap_report(splits).to_string(index=False))
print()
print("accuracy ceiling implied by contradictory duplicates:",
      leakage.annotation_ceiling(splits, "test"))

contra = leakage.contradictory_examples(splits, "train", "test")
print(f"\n{len(contra)} tweets appear verbatim in train and test with different gold labels:")
contra.head(10)

### 1.2 figures for the report

In [ ]:
viz.class_distribution(splits)
viz.length_distribution(splits, max_len=128)
plt.show()

## 2 how the tokenizers segment kinyarwanda

an africa adapted model is supposed to split the language better that is easy to
check so check it before leaning on it

In [ ]:
from src.tokenizer_analysis import fertility_table, segmentation_examples

print(fertility_table(splits["train"]["text"].tolist(), max_len=128).to_string(index=False))
print()
print(segmentation_examples().to_string(index=False))

## 3 baselines

the majority class one shows what doing nothing scores the tfidf logistic
regression one is what the transformer has to beat

In [ ]:
from src.baseline import BASELINES, run_baseline, top_features

fitted, baseline_val = {}, {}
for name in BASELINES:
    model, m, secs = run_baseline(name, splits, eval_split="validation")
    fitted[name], baseline_val[name] = model, m
    print(format_metrics_report(m, name), f"\n  fit: {secs:.2f}s\n")

In [ ]:
# shows the model learned real kinyarwanda sentiment words
for label, feats in top_features(fitted["TF-IDF + LogReg"], n=10).items():
    print(f"{label:9s}: {', '.join(w for w, _ in feats)}")

## 4 baseline test scores

nothing here was tuned the baselines use their library defaults so scoring them on
the test set once cannot bend any choice made later

In [ ]:
for name in BASELINES:
    model, m, _ = run_baseline(name, splits, eval_split="test")
    save_metrics(m, name, "test")
    print(f"{name:24s} macro-F1 {m['macro_f1']:.4f}  weighted-F1 {m['weighted_f1']:.4f}  acc {m['accuracy']:.4f}")

BASELINE_VAL_F1 = baseline_val["TF-IDF + LogReg"]["macro_f1"]
print(f"\nbar to beat on validation: macro-F1 {BASELINE_VAL_F1:.4f}")

## 5 transformer experiments  *gpu required*

each run states its hypothesis first and then goes in `reports/results/experiments_*.csv`
validation scores only no test set in this section

In [ ]:
import torch, transformers
print("transformers:", transformers.__version__)

assert int(transformers.__version__.split(".")[0]) < 5, \
    "needs transformers 4.x re-run the setup cell then Runtime > Restart session"
assert torch.cuda.is_available(), \
    "no gpu Runtime > Change runtime type > T4 GPU then re-run from the top"

print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from src.train_transformer import TrainConfig, run_experiment, predict, AFRO_XLMR_BASE, XLMR_BASE
from src.experiment_log import reset_log, load_all_experiments

# rerunning this notebook should not duplicate rows in the log
for m in ("afro-xlmr-base", "xlm-roberta-base"):
    reset_log(m)

runs = {}

In [ ]:
# if this raises cuda out of memory halve batch_size to 8 everything else
# stays comparable because only the optimiser step size changes not the data

# e1  the reference configuration
runs["E1"] = run_experiment(
    "E1", TrainConfig(label="E1-afroxlmr-default"), splits,
    change="AfroXLMR-base, lr 2e-5, 3 epochs, max_len 128, batch 16",
    hypothesis="A model with Kinyarwanda in its pretraining beats TF-IDF+LogReg on macro-F1",
    decision="", reason="")

In [ ]:
# e2  learning rate two runs one below one above the convention
for lr, tag in [(1e-5, "E2a"), (5e-5, "E2b")]:
    runs[tag] = run_experiment(
        tag, TrainConfig(learning_rate=lr, label=f"{tag}-lr{lr:g}"), splits,
        change=f"learning rate {lr:g} instead of 2e-5",
        hypothesis="2e-5 is the RoBERTa-family convention; 1e-5 may underfit in 3 epochs and 5e-5 may destabilise a 3.3k-example fine-tune",
        decision="", reason="")

In [ ]:
# e3  more epochs load_best_model_at_end means a worse final epoch is not kept
# so this measures whether an extra pass finds anything not whether it overfits
runs["E3"] = run_experiment(
    "E3", TrainConfig(epochs=5, label="E3-5epochs"), splits,
    change="5 epochs instead of 3",
    hypothesis="3.3k examples is small; extra epochs will overfit rather than help",
    decision="", reason="")

In [ ]:
# e4  shorter sequences section 2 measured that 5% of tweets exceed 64 subwords
runs["E4"] = run_experiment(
    "E4", TrainConfig(max_len=64, label="E4-maxlen64"), splits,
    change="max_len 64 instead of 128",
    hypothesis="Truncating ~5% of tweets costs little and roughly halves training time",
    decision="", reason="")

In [ ]:
# e5  the ablation same everything base model swapped section 2 established the
# two share a vocabulary so this difference is attributable to pretraining alone
runs["E5"] = run_experiment(
    "E5", TrainConfig(model_name=XLMR_BASE, label="E5-xlmr-ablation"), splits,
    change="plain XLM-RoBERTa-base in place of AfroXLMR-base, all else identical",
    hypothesis="AfroXLMR's African-language adaptive pretraining is worth a measurable amount of macro-F1",
    decision="", reason="")

In [ ]:
# e6  preprocessing reloads the data with the heavier cleaning flags on
splits_heavy = load_splits(strip_emoji=True, fix_elongation=True)
runs["E6"] = run_experiment(
    "E6", TrainConfig(label="E6-heavy-cleaning"), splits_heavy,
    change="emoji stripped and elongated characters collapsed",
    hypothesis="Emoji are often the only sentiment marker in a code-mixed tweet, so removing them should LOWER macro-F1",
    decision="", reason="")

In [ ]:
# e7  class weighting matching the baseline's class_weight="balanced"
runs["E7"] = run_experiment(
    "E7", TrainConfig(class_weights=True, label="E7-class-weighted"), splits,
    change="inverse-frequency class weights in the cross-entropy loss",
    hypothesis="Up-weighting the smaller positive class raises macro-F1 even if accuracy drops",
    decision="", reason="")

### 5.1 the experiment table

In [ ]:
log = load_all_experiments()
display(log[["exp_id", "stage", "change", "hypothesis", "val_macro_f1", "val_weighted_f1", "val_accuracy", "train_seconds"]])

viz.experiment_results(log, baseline_f1=BASELINE_VAL_F1)
plt.show()

best_id = log.loc[log["val_macro_f1"].idxmax(), "exp_id"]
print(f"\nbest on validation: {best_id}")
print("Apply the tie rule from src/experiment_log.py before declaring a winner: "
      "configurations within 0.01 macro-F1 are tied, and the cheaper one wins.")

## 6 final test evaluation  *run once*

pick the winner by the tie rule above set `BEST` to its key in `runs` and run this
section once nothing after this can change a configuration choice

In [ ]:
BEST = "E1"   # < set from section 5.1
best = runs[BEST]
cfg, trainer, tokenizer = best["cfg"], best["trainer"], best["tokenizer"]
print("final model:", cfg.summary())

test_preds, test_probs = predict(trainer, splits, "test", cfg.max_len, tokenizer)
m_test = compute_metrics(splits["test"]["label"], test_preds)
save_metrics(m_test, f"AfroXLMR ({BEST})", "test")
print(format_metrics_report(m_test, f"AfroXLMR ({BEST})"))

In [ ]:
# leak free score the same predictions with the 27 duplicate train rows removed
mask = leakage.clean_mask(splits, "test").to_numpy()
m_clean = compute_metrics(splits["test"]["label"][mask], test_preds[mask])
save_metrics(m_clean, f"AfroXLMR ({BEST}) clean", "test")
print(f"full test  ({len(test_preds)} rows): macro-F1 {m_test['macro_f1']:.4f}")
print(f"clean test ({int(mask.sum())} rows): macro-F1 {m_clean['macro_f1']:.4f}")
print("A large gap would mean the score leans on memorised duplicates.")

In [ ]:
plot_confusion_matrix(m_test["confusion_matrix"], f"AfroXLMR ({BEST})", normalise=True)
plt.show()
comparison_table()

## 7 error analysis

reading a few failures only confirms what you already believed so every guess about
why the model fails is tested by comparing error rates and only the ones the data
supports go in the report

In [ ]:
from src import error_analysis as ea

frame = ea.build_frame(splits["test"], test_preds, test_probs, model_name=f"AfroXLMR ({BEST})")
paths = ea.export(frame, f"afroxlmr_{BEST}")

rates = ea.feature_error_rates(frame)
display(rates)
viz.error_feature_chart(rates, 1 - frame["correct"].mean())
plt.show()

In [ ]:
print("Which classes get confused with which:")
display(ea.confusion_pairs(frame))

print("\nMost CONFIDENT errors -- what the model has learned wrongly:")
display(frame[~frame["correct"]].nlargest(10, "confidence")[
    ["raw_text", "true_label", "pred_label", "confidence"]])

print("\nMost confident CORRECT predictions -- the 'performs well' examples:")
display(frame[frame["correct"]].nlargest(5, "confidence")[
    ["raw_text", "true_label", "confidence"]])

In [ ]:
# where the two models disagree the clearest evidence of what pretraining bought
import numpy as np
base_preds = fitted["TF-IDF + LogReg"].predict(splits["test"]["text"])
base_probs = fitted["TF-IDF + LogReg"].predict_proba(splits["test"]["text"])
base_frame = ea.build_frame(splits["test"], base_preds, base_probs, model_name="TF-IDF + LogReg")

disagree = ea.disagreement(base_frame, frame, "baseline", "transformer")
disagree.to_csv("reports/results/disagreement_baseline_vs_transformer.csv", index=False)
display(disagree.head(12))

## 8 publish the model

pushes the weights to the hub so the gradio app in `app/` loads them by name the app
holds no weights itself so the space and this notebook run the same model

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HUB_ID = "mbonikevin/afroxlmr-kinyarwanda-sentiment"   # < set this

trainer.model.push_to_hub(HUB_ID)
tokenizer.push_to_hub(HUB_ID)
print(f"https://huggingface.co/{HUB_ID}")
print("Set MODEL_ID in app/app.py to this, then deploy the Space.")